# Task 1: Data Collection and Preprocessing

## Objective
Scrape user reviews from Google Play Store banking apps and preprocess them into a clean analysis-ready dataset.

## Banks Included
- Commercial Bank of Ethiopia (CBE)
- Bank of Abyssinia (BOA)
- Dashen Bank

## Dataset Fields
- Review Text
- Rating
- Review Date
- Bank Name
- Source

## Expected Output
A cleaned CSV dataset ready for sentiment and thematic analysis.

In [2]:
%pip install google-play-scraper

Defaulting to user installation because normal site-packages is not writeable
  Using cached google_play_scraper-1.2.7-py3-none-any.whl.metadata (50 kB)
Using cached google_play_scraper-1.2.7-py3-none-any.whl (28 kB)
Note: you may need to restart the kernel to use updated packages.


Import Libraries

In [4]:
import pandas as pd
import numpy as np

from google_play_scraper import reviews, Sort

import warnings
warnings.filterwarnings("ignore")

## Define Google Play Store App IDs

Each bank application on the Google Play Store has a unique application ID.

These IDs are required to scrape user reviews programmatically.

In [5]:
apps = {
    "CBE": "com.combanketh.mobilebanking",
    "BOA": "com.boa.boaMobileBanking",
    "Dashen": "com.dashen.dashensuperapp"
}

apps

{'CBE': 'com.combanketh.mobilebanking',
 'BOA': 'com.boa.boaMobileBanking',
 'Dashen': 'com.dashen.dashensuperapp'}

In [6]:
all_reviews = []

for bank, app_id in apps.items():

    print(f"Scraping reviews for {bank}...")

    result, _ = reviews(
        app_id,
        lang='en',
        country='et',
        sort=Sort.NEWEST,
        count=400
    )

    for review in result:
        all_reviews.append({
            "review": review["content"],
            "rating": review["score"],
            "date": review["at"],
            "bank": bank,
            "source": "Google Play"
        })

print("Scraping completed.")

Scraping reviews for CBE...
Scraping reviews for BOA...
Scraping reviews for Dashen...
Scraping completed.


Create DataFrame

In [7]:
df = pd.DataFrame(all_reviews)

df.head()

,review,rating,date,bank,source
0,this update got crazy i don't know what's goin...,1,2026-05-15 23:20:32,CBE,Google Play
1,thanks for you 😘,5,2026-05-15 20:11:22,CBE,Google Play
2,it's okay,4,2026-05-15 19:53:26,CBE,Google Play
3,It's not allowing me to transfer money.,2,2026-05-15 12:22:49,CBE,Google Play
4,IT'S NOT WORK ON HUAWEI DEVICES,4,2026-05-15 12:07:21,CBE,Google Play


In [8]:
print("Dataset Shape:")
print(df.shape)

Dataset Shape:
(1200, 5)


Missing Value Analysis

In [9]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
review    0
rating    0
date      0
bank      0
source    0
dtype: int64


 there is no missing value

In [10]:
print("Duplicate Rows:")
print(df.duplicated().sum())


Duplicate Rows:
0


### Duplicate Review Analysis

Duplicate inspection showed that the dataset contains **0 duplicate rows**.

This confirms that all collected reviews are unique and no additional
deduplication logic was required beyond validation.

In [12]:
# Remove duplicates
df = df.drop_duplicates()

# Remove missing reviews and ratings
df = df.dropna(subset=["review", "rating"])

# Normalize dates
df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")

print("Cleaning completed.")

Cleaning completed.


In [13]:
print(df.info())

df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   review  1200 non-null   object
 1   rating  1200 non-null   int64 
 2   date    1200 non-null   object
 3   bank    1200 non-null   object
 4   source  1200 non-null   object
dtypes: int64(1), object(4)
memory usage: 47.0+ KB
None


,review,rating,date,bank,source
0,this update got crazy i don't know what's goin...,1,2026-05-15,CBE,Google Play
1,thanks for you 😘,5,2026-05-15,CBE,Google Play
2,it's okay,4,2026-05-15,CBE,Google Play
3,It's not allowing me to transfer money.,2,2026-05-15,CBE,Google Play
4,IT'S NOT WORK ON HUAWEI DEVICES,4,2026-05-15,CBE,Google Play


In [14]:
df["bank"].value_counts()

bank
CBE       400
BOA       400
Dashen    400
Name: count, dtype: int64

In [15]:
df["rating"].value_counts().sort_index()

rating
1    242
2     34
3     58
4     88
5    778
Name: count, dtype: int64

# Rating Distribution Analysis

The rating distribution indicates that most users had a positive experience with the banking applications.

## Observations

- 5-star ratings dominate the dataset with 778 reviews.
- 1-star ratings are also significant with 242 reviews, showing that some users experienced serious issues.
- 2-star and 3-star reviews are relatively low.
- The dataset reflects a generally positive perception of the banking apps, but there are still noticeable customer pain points that require investigation.

## Interpretation

High 5-star ratings may indicate:
- Smooth transaction experiences
- Easy-to-use interfaces
- Reliable mobile banking services

Low ratings may indicate:
- Login issues
- OTP failures
- App crashes
- Slow performance during transactions

These findings will be explored further in Task 2 using sentiment and thematic analysis.

In [18]:
output_path = "../data/bank_reviews_clean.csv"

df.to_csv(output_path, index=False)

print(f"Dataset saved to: {output_path}")

Dataset saved to: ../data/bank_reviews_clean.csv


In [19]:
clean_df = pd.read_csv("../data/bank_reviews_clean.csv")

clean_df.head()

,review,rating,date,bank,source
0,this update got crazy i don't know what's goin...,1,2026-05-15,CBE,Google Play
1,thanks for you 😘,5,2026-05-15,CBE,Google Play
2,it's okay,4,2026-05-15,CBE,Google Play
3,It's not allowing me to transfer money.,2,2026-05-15,CBE,Google Play
4,IT'S NOT WORK ON HUAWEI DEVICES,4,2026-05-15,CBE,Google Play


### Dataset Export

The cleaned dataset was successfully exported as:

`bank_reviews_clean.csv`

This file will serve as the primary input dataset for:
- Sentiment Analysis
- Thematic Analysis
- Visualization
- Database Engineering

### Cleaned Dataset Overview

The cleaned dataset contains:

- 1,200 observations
- 5 structured columns
- No missing values

Column descriptions:

| Column | Description |
|---|---|
| review | User review text |
| rating | Star rating (1–5) |
| date | Review posting date |
| bank | Bank/app identifier |
| source | Data source (Google Play) |

The dataset structure aligns with the Week 2 challenge requirements.

In [20]:
print("Final Dataset Shape:", clean_df.shape)

print("\nMissing Values:")
print(clean_df.isnull().sum())

print("\nReviews Per Bank:")
print(clean_df["bank"].value_counts())

Final Dataset Shape: (1200, 5)

Missing Values:
review    0
rating    0
date      0
bank      0
source    0
dtype: int64

Reviews Per Bank:
bank
CBE       400
BOA       400
Dashen    400
Name: count, dtype: int64
